# Movie Sentiment Analysis MLOps - Notebook Experiments

This notebook provides an interactive environment to load, preprocess, train, evaluate, and predict sentiment using our DistilBERT pipeline.

### 1. Setup Environment and Paths
We need to add the parent directory to the system path so that we can run relative imports easily.

In [ ]:
import sys
from pathlib import Path

# Ensure project root is in system path
project_root = Path("").resolve().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

print(f"Project root: {project_root}")

### 2. Loading and Preprocessing Data
Let's import our load function from `preprocess.py` and inspect the dataset splits.

In [ ]:
from src.preprocess import load_imdb_data

# Load a small subset of 100 samples for inspection
train_df, val_df, test_df = load_imdb_data(max_samples=100)

print(f"Training set rows: {len(train_df)}")
print(f"Validation set rows: {len(val_df)}")
print(f"Test set rows: {len(test_df)}")
print("\nSample positive review:")
print(train_df[train_df["label"] == 1].iloc[0]["text"][:150] + "...")
print("\nSample negative review:")
print(train_df[train_df["label"] == 0].iloc[0]["text"][:150] + "...")

### 3. Initialize Predictor (Untrained/Base fallback)
Let's test our `SentimentPredictor` class. If no fine-tuned weights exist yet, it will log a warning and fall back to the raw, pre-trained `distilbert-base-uncased` model for structural execution.

In [ ]:
from src.predict import SentimentPredictor

predictor = SentimentPredictor()
review = "This film has some of the most beautiful shots I have ever seen."
sentiment, confidence = predictor.predict(review)

print(f"Review: '{review}'")
print(f"Predicted Sentiment: {sentiment}")
print(f"Confidence: {confidence:.4f}")

### 4. Running the Model Training
We can trigger our training script directly. This executes the PyTorch fine-tuning loop and logs parameters, training loss, validation metrics, and model registry artifacts to **MLflow**.

In [ ]:
import os

# Set small values for training in notebook to ensure it runs quickly on CPU
os.environ["MAX_SAMPLES"] = "20"
os.environ["EPOCHS"] = "1"
os.environ["BATCH_SIZE"] = "4"

from src.train import main as run_training

print("Starting training run...")
run_training()

### 5. Running the Model Evaluation
Now let's run the model evaluation script which calculates metrics, prints a classification report, and logs the confusion matrix to MLflow.

In [ ]:
from src.evaluate import run_evaluation

metrics = run_evaluation()
print("\nEvaluation Metrics:")
for k, v in metrics.items():
    print(f"{k.capitalize()}: {v:.4f}")